<a href="https://colab.research.google.com/github/diishsoap/Text-Complexity-Analyser/blob/main/CLEARText_%E2%80%94_Text_Complexity_Analyser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CLEARText — Text Complexity Analyser:

A text analyser for students in specialised fields to understand if their reports are simple enough for non-specialised public to comprehend.

<br>

**Veritas AI International High School Hackathon 2026 · SOLO SUBMISSION**

**Track:** AI + Languages (built on my own idea)
**Team:** Kiran Dhara

**One-sentence pitch:** The CLEARText Complexity Analyser attempts to make specialised, technical knowledge more accessible by helping highschool and undergraduate students figure out if their academic report can be comprehended by an non-specialised, public audience.

**How to run:** Runtime → Run all. It takes under a minute and needs no API keys or uploads.



## 1. Loading the data


I used the CLEAR Corpus dataset by CommonLit (<a href="https://www.commonlit.org/blog/introducing-the-clear-corpus-an-open-dataset-to-advance-research-28ff8cfea84a/">about the Clear Corpus</a>): with 4724 total records and 40 info columns recording metadata, as well as readablity scores.

I loaded a public copy hosted on my GitHub so that the notebook can run without any downloads. The license agreement is present in the GitHub profile's README. The profile is shared in the write-up.

In [ ]:
import pandas as pd         #tabular data and csv
import numpy as np          #mathematical operations
import re                   #regular expression

# raw file version on github (license in github readme)
DATA_URL = "https://raw.githubusercontent.com/diishsoap/text-analyser/main/data/CLEAR%20Corpus%206.01%20-%20CLEAR%20Corpus%206.01.csv"

df = pd.read_csv(DATA_URL)

print("Original dataset:", df.shape)

Original dataset: (4724, 40)


In [ ]:
# making sure dataset imported properly

df.head()


,ID,Last Changed,Author,Title,Anthology,URL,Source,Pub Year,Category,Location,...,CAREC_M,CARES,CML2RI,firstPlace_pred,secondPlace_pred,thirdPlace_pred,fourthPlace_pred,fifthPlace_pred,sixthPlace_pred,Kaggle split
0,400,NaN,Carolyn Wells,Patty's Suitors,NaN,http://www.gutenberg.org/cache/epub/5631/pg563...,gutenberg,1914.0,Lit,mid,...,0.11952,0.457534,12.097815,-0.383831,-0.283604,-0.346879,-0.281620,-0.247767,-0.289945,Train
1,401,NaN,Carolyn Wells,Two Little Women on a Holiday,NaN,http://www.gutenberg.org/cache/epub/5893/pg589...,gutenberg,1917.0,Lit,mid,...,0.04921,0.462510,22.550179,-0.260307,-0.209960,-0.061565,-0.234231,-0.201347,-0.156156,Train
2,402,NaN,Carolyn Wells,Patty Blossom,NaN,http://www.gutenberg.org/cache/epub/20945/pg20...,gutenberg,1917.0,Lit,mid,...,0.09724,0.369259,18.125279,-0.615037,-0.530600,-0.527847,-0.550180,-0.565762,-0.538852,Train
3,403,NaN,CHARLES KINGSLEY,THE WATER-BABIES\nA Fairy Tale for a Land-Baby,NaN,http://www.gutenberg.org/files/25564/25564-h/2...,gutenberg,1863.0,Lit,mid,...,0.08856,0.390759,10.959460,-1.528806,-1.525546,-1.471455,-1.265776,-1.422547,-1.393155,Test
4,404,NaN,Charles Kingsley,HOW THE ARGONAUTS WERE DRIVEN INTO THE UNKNOWN...,The Heroes\n or Greek Fairy Tales for my...,http://www.gutenberg.org/files/677/677-h/677-h...,gutenberg,1889.0,Lit,mid,...,0.08798,0.389226,3.195960,-1.335586,-1.321922,-1.163985,-1.122501,-1.185518,-1.271324,Train


## 2. Data inspection

Before training anything, I need to clean the data. I inspect and remove anything that isn't in the public domain, and make sure that none of my records include missing/duplicated data in the important columns. I also standardise the whitespace characters.

In [ ]:
#checking licenses

print(df["License"].value_counts(dropna=False))

License
PD                                        3194
CC BY 4.0                                  817
CC BY-SA 3.0                               306
CC BY-SA 3.0 and GFDL                      274
CC BY-NC-SA 2.0                             74
CC BY 3.0                                   16
CC BY-NC-SA 4.0                             11
CC BY-NC-SA 3.0                              8
CC BY-NC-ND 4.0                              6
CC BY-NC 3.0                                 4
CC BY-NC                                     4
CC BY NC-SA 2.0                              3
CC BY-NC 4.0                                 1
CC-BY-SA                                     1
CC BY-NC 2.0                                 1
CC-BY-NC                                     1
GNU Free Document License, version 1.2       1
GNU GPL                                      1
GNU FDL 1.2                                  1
Name: count, dtype: int64


In [ ]:
# eliminating records that are not public domain

df_clean = df[
    df['License'].str.strip() == "PD"
].copy()

print("original: ", len(df))
print("only pd: ", len(df_clean))

print(df_clean["License"].value_counts())

original:  4724
only pd:  3194
License
PD    3194
Name: count, dtype: int64


In [ ]:
# checking if any important values are missing
# observation:
# only optional metadata is missing, which won't be required anyway.
# so, nothing is eliminated

missing = df_clean.isnull().sum()

print(missing[missing > 0].sort_values(ascending=False))

Last Changed     3185
British Words    2864
Anthology        1003
Pub Year            2
dtype: int64


In [ ]:
#checking for duplicates. none found

print("duplicate ids: ", df_clean["ID"].duplicated().sum())

print("duplicate excerpts: ",
      df_clean["Excerpt"].duplicated().sum())


duplicate ids:  0
duplicate excerpts:  0


In [ ]:
# cleaning irregular whitespace
# by standardizing whitespace to a singular ' ' space

import re     # support for regular expressions

def clean_text(text):
    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df_clean["Processed_Excerpt"] = (
    df_clean["Excerpt"].apply(clean_text)
)

In [ ]:
# final run through of cleaned data.

print("Final shape:", df_clean.shape)

print("\nMissing values:")
print(df_clean.isnull().sum())

print("\nCategories:")
print(df_clean["Category"].value_counts())

print("\nReadability statistics:")
print(df_clean["BT Easiness"].describe())

Final shape: (3194, 41)

Missing values:
ID                                       0
Last Changed                          3185
Author                                   0
Title                                    0
Anthology                             1003
URL                                      0
Source                                   0
Pub Year                                 2
Category                                 0
Location                                 0
License                                  0
MPAA\nMax                                0
MPAA \n#Max                              0
MPAA\n#Avg                               0
Excerpt                                  0
Google\nWC                               0
Joon\nWC v1                              0
British WC                               0
British Words                         2864
Sentence\nCount v1                       0
Sentence\nCount v2                       0
Paragraphs                               0
BT Easiness  

In [ ]:
#saving and downloading clean data
'''
from google.colab import files

df_clean.to_csv("processed_data.csv", index=False)
files.download("processed_data.csv")
'''

#i used this to save the cleaned csv file, and uploaded it to GitHub.
#this step is not neccessary anymore as further downloads are not required

'\nfrom google.colab import files\n\ndf_clean.to_csv("processed_data.csv", index=False)\nfiles.download("processed_data.csv")\n'

In [ ]:
# checking a sample of original and processed excerpts
# to make sure that preliminary processing worked as intended

print("Original excerpt:")
print(df_clean["Excerpt"].iloc[0])

print("\nProcessed excerpt:")
print(df_clean["Processed_Excerpt"].iloc[0])

Original excerpt:
When the young people returned to the ballroom, it presented a decidedly changed appearance. Instead of an interior scene, it was a winter landscape.
The floor was covered with snow-white canvas, not laid on smoothly, but rumpled over bumps and hillocks, like a real snow field. The numerous palms and evergreens that had decorated the room, were powdered with flour and strewn with tufts of cotton, like snow. Also diamond dust had been lightly sprinkled on them, and glittering crystal icicles hung from the branches.
At each end of the room, on the wall, hung a beautiful bear-skin rug.
These rugs were for prizes, one for the girls and one for the boys. And this was the game.
The girls were gathered at one end of the room and the boys at the other, and one end was called the North Pole, and the other the South Pole. Each player was given a small flag which they were to plant on reaching the Pole.
This would have been an easy matter, but each traveller was obliged to wear 

## 3. Split into training and test sets

I held back 20% of the messages for testing and did not train on them.

In [ ]:
# checking how many unique titles and sources exist.
print("Unique titles:", df_clean["Title"].nunique())
print("Unique sources:", df_clean["Source"].nunique())

#checking how frequently titles occur.
print(df_clean["Title"].value_counts().head(15))

Unique titles: 3144
Unique sources: 6
Title
Invention and Discovery                 13
?                                        8
Current History                          4
LITTLE MISCHIEF                          3
THE ENCHANTED CAVE                       2
THE THREE BEARS                          2
Queen Victoria                           2
Progress Invention and Discovery         2
Latest News                              2
Rainbow Valley                           2
A Little Princess                        2
The Secret Garden                        2
The Adventures of Lightfoot the Deer     2
THE LITTLE MATCH GIRL                    2
THE UGLY DUCKLING                        2
Name: count, dtype: int64


In [ ]:
# splitting the data into 80:20 train/test sets

from sklearn.model_selection import GroupShuffleSplit

# separating model inputs, target, and grouping variable
x = df_clean["Processed_Excerpt"]
y = df_clean["BT Easiness"]
groups = df_clean["Title"].fillna("Unknown")

# creating an 80:20 grouped train/test split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(x, y, groups=groups)
)

# assigning training and testing data
x_train = x.iloc[train_idx]
x_test = x.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training observations:", len(x_train))
print("Testing observations:", len(x_test))

print("Unique training titles:", groups.iloc[train_idx].nunique())
print("Unique testing titles:", groups.iloc[test_idx].nunique())

# verifying that no titles overlap
print(
    "Overlapping titles:",
    len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))
)


# split is ~80.2% and ~19.8% (there are some different excerpts that are from the same title)

Training observations: 2563
Testing observations: 631
Unique training titles: 2515
Unique testing titles: 629
Overlapping titles: 0


## 4. Baseline

This is the simplest possible rule. Any model we build has to beat it.

The baseline is the mean BT Easiness score of the training dataset, hence providing a simple starting point/baseline for our actually trained model to outperform.

In [ ]:
## establishing the baseline


from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np  # alr imported, here for ease of later comprehension / debug

# creating and training the baseline model
baseline = DummyRegressor(strategy="mean")
baseline.fit(x_train.to_frame(), y_train)   # fit establishes avg target value

# generating predictions on the test set
# predict() generates predictions for all passages
baseline_predictions = baseline.predict(x_test.to_frame())


# evaluating baseline performance
# MAE measures avg absolute diff btw actual n predicted scores
# RMSE penalises larger predictions more easily
# R² measures how much variation in the target the model accounts for,
#      relative to predicting the test-set mean

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))
baseline_r2 = r2_score(y_test, baseline_predictions)

print("baseline results:")
print("MAE:", baseline_mae)
print("RMSE:", baseline_rmse)
print("R²:", baseline_r2)

baseline results:
MAE: 0.8383773383932989
RMSE: 1.0287946704470636
R²: -0.0005074172799639509


## 5. Model 1: TF-IDF + ridge regression

TF-IDF turns text into numerical values. Ridge Regression then learns the relationship between those features and human-derived BT Easiness scores.

Both steps are then combined into a Scikit-learn pipeline to ensure consistent preprocessing during training and prediction.

In [ ]:
## ridge regression implementation

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

# creating a pipeline combining text vectorisation and regression
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2), #uses individual and paired words
        min_df=2, #ignores terms appearing in <2 passages
        max_df=0.95, #ignores terms in >95% passages
        max_features=20000, #limits vocab to 20k features
        sublinear_tf=True #controls the strength of ridge regularisation
    )),
    ("regressor", Ridge(alpha=1.0))
])

In [ ]:
# training the pipeline using the training data
model.fit(x_train, y_train)

# generating predictions for the unseen test data
model_predictions = model.predict(x_test)

In [ ]:
# model evaluation

# calculating model evaluation metrics
model_mae = mean_absolute_error(y_test, model_predictions)
model_rmse = np.sqrt(mean_squared_error(y_test, model_predictions))
model_r2 = r2_score(y_test, model_predictions)

print("Ridge Regression Results")
print("MAE:", model_mae)
print("RMSE:", model_rmse)
print("R²:", model_r2)

Ridge Regression Results
MAE: 0.5374749289543969
RMSE: 0.6758740442566593
R²: 0.5681882676176424


## 6. Comparing the results of Model 1 and the baseline model

I used pandas to create a comparison table so we can see if there's been any improvement in the predictions.

In [ ]:
#creating a comparison table
#for baseline and trained model evaluations

import pandas as pd

results = pd.DataFrame({
    "Model": ["Mean Baseline", "TF-IDF + Ridge"],
    "MAE": [baseline_mae, model_mae],
    "RMSE": [baseline_rmse, model_rmse],
    "R²": [baseline_r2, model_r2]
})

print(results.to_string(index=False))

         Model      MAE     RMSE        R²
 Mean Baseline 0.838377 1.028795 -0.000507
TF-IDF + Ridge 0.537475 0.675874  0.568188


### Interpretation of compared scores:


<br>

Lower MAE: predictions are closer to the actual scores on average
Lower RMSE: fewer large prediction errors
Higher R²: more variation in the actual scores is accounted for.

As we can see, the newly trained model outperforms the baseline across all three metrics.

In particular, MAE decreased by ~35.9%, RMSE decreased by ~34.3% and the model now accounts for ~56.8% of the variation in BT easiness scores in the test set.

This demonstrates how the model learnt useful patterns from its training rather than regurgitating the average score.

In [ ]:
#comparing actual and predicted values for a sample of test passages

comparison = pd.DataFrame({
    "Actual BT Easiness": y_test,
    "Predicted BT Easiness": model_predictions
})

comparison["Absolute Error"] = abs(
    comparison["Actual BT Easiness"] -
    comparison["Predicted BT Easiness"]
)

print(comparison.head(10))

    Actual BT Easiness  Predicted BT Easiness  Absolute Error
2            -0.580118              -0.977880        0.397762
5             0.247197               0.311406        0.064208
6            -0.861809              -1.345253        0.483445
15            0.245806              -0.124249        0.370055
26           -1.566220              -0.982590        0.583630
29            0.633139              -0.710684        1.343823
32           -0.146083              -0.760103        0.614019
37            0.404633              -0.674807        1.079440
38           -0.952123              -0.431854        0.520269
39           -1.053898              -0.844702        0.209195


In [ ]:
def analyse_text(text):

    # generating a BT Easiness prediction
    predicted_score = model.predict([text])[0]

    # calculating its relative position in the training distribution
    percentile = np.mean(y_train <= predicted_score) * 100

    # assigning a relative difficulty description
    if percentile >= 75:
        difficulty = "relatively easier"
    elif percentile <= 25:
        difficulty = "relatively more difficult"
    else:
        difficulty = "moderate relative difficulty"


    word_count = len(text.split()) #calculates basic text statistics

    print("Text Analysis")
    print("-" * 35)
    print(f"Predicted BT Easiness: {predicted_score:.3f}")
    print(f"Relative percentile: {percentile:.1f}%")
    print(f"Estimated difficulty: {difficulty}")
    print(f"Word count: {word_count}")

## 7. Error analysis

Or, where does the model fail?

In [ ]:
# this sample text has been randomly chosen from the processed dataset

sample_text = """
When a whaling ship is beset in the ice of Davis Straits, there is little work for her second engineer, once the engines have been nicely tallowed down. Now, I am no man that can sit in his berth and laze. If I've no work to do, I get a-thinking about my home at Ballindrochater and the ministry, which my father intended I should have adorned, and what a fool I've made of myself, and this is depressing. I was not over-popular already on the Gleaner on account of some prophecies I had made in anger, which had unfortunately come true. The crew, and the captain, too, had come to fear my prophetic powers.
"""

analyse_text(sample_text)

Text Analysis
-----------------------------------
Predicted BT Easiness: -1.604
Relative percentile: 31.9%
Estimated difficulty: moderate relative difficulty
Word count: 114


In [ ]:
# comparing the external passage's actual score
# with its position in the training-set distribution.

actual_score = -2.441540897       #of the earlier sample text
predicted_score = -1.604

actual_percentile = np.mean(y_train <= actual_score) * 100

print("Actual score:", actual_score)
print("Predicted score:", predicted_score)
print("Absolute error:", abs(actual_score - predicted_score))
print("Actual percentile:", actual_percentile)

Actual score: -2.441540897
Predicted score: -1.604
Absolute error: 0.8375408969999998
Actual percentile: 10.92469761997659


In [ ]:
error_analysis = pd.DataFrame({
    "actual": y_test,
    "predicted": model_predictions
})

error_analysis["residual"] = error_analysis["actual"] - error_analysis["predicted"]
error_analysis["absolute_error"] = error_analysis["residual"].abs()

error_analysis["difficulty_group"] = pd.qcut(
    error_analysis["actual"],
    q=4,
    labels=["Most difficult", "Moderately difficult", "Moderately easy", "Easiest"]
)

error_analysis.groupby("difficulty_group", observed=True)["absolute_error"].agg(
    ["count", "mean", "median", "max"]
)

,count,mean,median,max
difficulty_group,,,,
Most difficult,158,0.670579,0.612454,1.818981
Moderately difficult,158,0.383407,0.320415,1.305312
Moderately easy,157,0.389172,0.337673,1.583416
Easiest,158,0.705803,0.609563,2.573020


In [ ]:
print("Actual score range:", y_test.min(), "to", y_test.max())
print("Predicted score range:", model_predictions.min(), "to", model_predictions.max())

print("Actual mean:", y_test.mean())
print("Predicted mean:", model_predictions.mean())

print("Actual standard deviation:", y_test.std())
print("Predicted standard deviation:", model_predictions.std())


# from here, we can see that the predicted std is significantly lower
# ie, it is predicting a narrower range of scores
# we can see this in how boths extremes are predicted very inconsistently and incorrectly


Actual score range: -3.676267773 to 1.546966393
Predicted score range: -2.721493216651785 to 0.48593157729648895
Actual mean: -1.0900911172076069
Predicted mean: -1.118119582713281
Actual standard deviation: 1.0293497286017732
Predicted standard deviation: 0.6716938004523921


In [ ]:
error_analysis.sort_values(
    "absolute_error",
    ascending=False
).head(10)

,actual,predicted,residual,absolute_error,difficulty_group
3193,1.546966,-1.026054,2.573020,2.573020,Easiest
2043,1.163127,-0.745309,1.908436,1.908436,Easiest
3205,1.316493,-0.527865,1.844358,1.844358,Easiest
1865,-3.668360,-1.849379,-1.818981,1.818981,Most difficult
2852,0.004187,-1.796292,1.800480,1.800480,Easiest
375,0.687905,-1.109808,1.797713,1.797713,Easiest
4212,-2.802928,-1.044376,-1.758552,1.758552,Most difficult
3531,-3.509585,-1.861985,-1.647599,1.647599,Most difficult
3045,-2.846767,-1.214803,-1.631964,1.631964,Most difficult
3270,0.381140,-1.247649,1.628790,1.628790,Easiest


In [ ]:
error_analysis["Category"] = df_clean.loc[y_test.index, "Category"]

error_analysis.groupby("Category")["absolute_error"].agg(
    ["count", "mean", "median", "max"]
)


,count,mean,median,max
Category,,,,
Info,233,0.554019,0.452136,1.908436
Lit,398,0.527789,0.440127,2.573020


Category doesnt seem to be a problem because their scores are relatively equal/similar. Hence, separate models for each category are unnecessary.

<br>

Instead, the model is struggling to accurately predict values closer to the extremes, whereas the moderately easy and difficult difficulty levels both seem to be doing much better.

<br>

Hence, the 1st model has a limitation of overestimating he easier passages and underestimating the harder passages due to a tendency to stick close to the mean, though not as close as the baseline.

## 8. Attempting to make a 2nd model

This model will be oriented towards reducing the tendency to stick close to the mean.

In [ ]:
char_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        min_df=2,
        max_features=30000,
        sublinear_tf=True
    )),
    ("regressor", Ridge(alpha=1.0))
])

char_model.fit(x_train, y_train)
char_predictions = char_model.predict(x_test)

print("Character-level results")
print("-" * 35)
print("MAE:", mean_absolute_error(y_test, char_predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test, char_predictions)))
print("R²:", r2_score(y_test, char_predictions))


# RIDGE REGRESSION RESULTS
# MAE: 0.5374749289543969
# RMSE: 0.6758740442566593
# R²: 0.5681882676176424


## above are the originals from the 1st model
#  as you can see, both MAE and RMSE have improved, although modestly,
#  as both have gotten lower in the 2nd model (this one)
#  R² has also increased
## hence, there has been a small improvement across all metrics

Character-level results
-----------------------------------
MAE: 0.5281953437033283
RMSE: 0.6541181104595664
R²: 0.5955403067332824


In [ ]:
char_error_analysis = pd.DataFrame({
    "actual": y_test,
    "predicted": char_predictions
})

char_error_analysis["residual"] = (
    char_error_analysis["actual"] - char_error_analysis["predicted"]
)

char_error_analysis["absolute_error"] = (
    char_error_analysis["residual"].abs()
)

char_error_analysis["difficulty_group"] = pd.qcut(
    char_error_analysis["actual"],
    q=4,
    labels=[
        "Most difficult",
        "Moderately difficult",
        "Moderately easy",
        "Easiest"
    ]
)

print("Actual standard deviation:", y_test.std())
print("Character prediction standard deviation:", char_predictions.std())
print(" ")

print(
    char_error_analysis.groupby(
        "difficulty_group",
        observed=True
    )["absolute_error"].agg(["count", "mean", "median", "max"])
)

Actual standard deviation: 1.0293497286017732
Character prediction standard deviation: 0.7432977280513525
 
                      count      mean    median       max
difficulty_group                                         
Most difficult          158  0.635507  0.588260  2.182599
Moderately difficult    158  0.407148  0.337711  1.583442
Moderately easy         157  0.436434  0.387485  1.642860
Easiest                 158  0.633111  0.591704  2.223542


In [ ]:
'''
count	mean	median	max
difficulty_group
Most difficult	158	0.670579	0.612454	1.818981
Moderately difficult	158	0.383407	0.320415	1.305312
Moderately easy	157	0.389172	0.337673	1.583416
Easiest	158	0.705803	0.609563	2.573020
'''

'\ncount\tmean\tmedian\tmax\ndifficulty_group\nMost difficult\t158\t0.670579\t0.612454\t1.818981\nModerately difficult\t158\t0.383407\t0.320415\t1.305312\nModerately easy\t157\t0.389172\t0.337673\t1.583416\nEasiest\t158\t0.705803\t0.609563\t2.573020\n'

The above code was the table for the 1st model. As we can see, the character-level TF-IDF model (Model 2) performs better overall, although its improvements are not consistent across every difficulty group.

- Most difficult: MAE decreases from 0.671 to 0.636, demonstrating improved predictions for the hardest passages.
- Moderately difficult: MAE increases from 0.383 to 0.407, showing a slight decline in performance.
- Moderately easy: MAE increases from 0.389 to 0.436, also indicating a decline.
- Easiest: MAE decreases from 0.706 to 0.633, showing improved predictions for the easiest passages.

In other words, Model 2 has improved in its predictions for the extremes, but has declined in accuracy slightly for the means.

The standard deviation also increases, bringing the range of values accounted for in Model 2 closer to that of the actual range, hence displaying an improvement.

Overall, Model 2 is the stronger model, supported by its lower MAE and RMSE and higher R². However, the improvements should be considered modest rather than substantial, as not all metrics have improved.

The preliminary model has several limitations. Its predictions tend to be closer to the mean than the actual scores, making particularly easy or difficult passages harder to estimate accurately.

TF-IDF also relies on character patterns rather than understanding meaning, context, sentence structure or semantic relationships. Furthermore, the dataset represents a limited collection of passages and may not generalise equally well to all types of writing.

The preliminary version of CLEARText demonstrates that machine learning can identify patterns in written text and use them to estimate relative readability.

While the model does not directly measure comprehension or guarantee how easily a reader will understand a passage, it provides a useful starting point for readability analysis. Future development could explore additional linguistic features, improved text representations and more diverse training data to improve prediction accuracy.

## 9. Creating an analyse_text function for Model 2

According to the improvement seen in Model 2, the text analysis of the same sample text must also improve. Here, I define analyse_text2 as a function for Model 2 and test it on the same sample as earlier to draw conclusions about the second model and possible future improvements/refinements.

In [ ]:
def analyse_text2(text):

    # generating a BT Easiness prediction
    predicted_score = char_model.predict([text])[0]

    # calculating its relative position in the training distribution
    percentile = np.mean(y_train <= predicted_score) * 100

    # assigning a relative difficulty description
    if percentile >= 75:
        difficulty = "relatively easier"
    elif percentile <= 25:
        difficulty = "relatively more difficult"
    else:
        difficulty = "moderate relative difficulty"


    word_count = len(text.split()) #calculates basic text statistics

    print("Text Analysis")
    print("-" * 35)
    print(f"predicted BT Easiness: {predicted_score:.3f}")
    print(f"relative percentile: {percentile:.1f}%")
    print(f"estimated difficulty: {difficulty}")
    print(f"word count: {word_count}")

In [ ]:
# this sample text has been randomly chosen from the refined dataset

sample_text2 = """
When a whaling ship is beset in the ice of Davis Straits, there is little work for her second engineer, once the engines have been nicely tallowed down. Now, I am no man that can sit in his berth and laze. If I've no work to do, I get a-thinking about my home at Ballindrochater and the ministry, which my father intended I should have adorned, and what a fool I've made of myself, and this is depressing. I was not over-popular already on the Gleaner on account of some prophecies I had made in anger, which had unfortunately come true. The crew, and the captain, too, had come to fear my prophetic powers.
"""

analyse_text2(sample_text2)

Text Analysis
-----------------------------------
predicted BT Easiness: -1.883
relative percentile: 23.7%
estimated difficulty: relatively more difficult
word count: 114


In [ ]:
# comparing the external passage's actual score
# with its position in the training-set distribution.

actual_score = -2.441540897       #of the earlier sample text
predicted_score = -1.883

actual_percentile = np.mean(y_train <= actual_score) * 100

print("Actual score:", actual_score)
print("Predicted score:", predicted_score)
print("Absolute error:", abs(actual_score - predicted_score))
print("Actual percentile:", actual_percentile)

Actual score: -2.441540897
Predicted score: -1.883
Absolute error: 0.5585408969999999
Actual percentile: 10.92469761997659


**Original error analysis using Model 1:**<br>
Actual score: -2.441540897<br>
Predicted score: -1.604<br>
Absolute error: 0.8375408969999998<br>
Actual percentile: 10.92469761997659<br>
<br><br>


Comparing this to the error analysis of Model 2, given above, we can see that there has been a clear improvement, though modest.

For the same passage, the character-level model reduced the absolute error from 0.838 to 0.559, an improvement of approximately 33.3%. Since lower BT Easiness scores indicate greater difficulty, Model 2 produced a prediction closer to the actual score and better captured the passage's relative difficulty.

This shows how the model can be improved further using more refinement. However, this is an isolated, single sample, so this might not display general improvement, but should be taken as an example.

## 10. Results summary

| | MAE | RMSE | R² |
|---|---|---|---|
| Baseline: mean BT Easiness score | 0.8383 | 1.0287 | -0.000 (zero deviation from mean / zero range) |
| Model 1: TF-IDF + ridge regression | 0.5375 (~36% improvement)	| 0.6759 (~34% improvement) | 0.5682 (57% improvement) |
| **Model 2: + character level TF-IDF** | **0.5282 (~1.73% improvement)** | **0.6541 (~3.23% improvement)** | **0.5955 (~4.81% improvement)** |

## 11. Limitations and next steps

- Both models aren't refined enough to accurately predict BT Easiness scores, yet, and though Model 2 improved at the extremes, its performance declined at the moderate difficulty levels
- These models can't be fully applied to a real life problem at the moment, but with further refinement, the model can be trained to accurately predict BT Easiness scores, and then used for completely unseen, user inputted passages using the `analyse_text` function.
- The model does not account for factors such as semantic complexity, contextual relationships, grammatical structures or figurative language, etc.<br>
So, future development could include more advanced training to include one or more of these aspects of readability, thus improving accuracy from a person's point of view, rather than that of a machine's.

<br>

## Sources
-  CLEAR (CommonLit Ease of Readability) Corpus: Crossley, S., Heintz, A., Choi, J.S. et al. A large-scaled corpus for assessing text readability. Behav Res 55, 491–507 (2023). https://doi.org/10.3758/s13428-022-01802-x <br><br>(Dataset is loaded into this notebook from a personal GitHub Repository, https://github.com/diishsoap/text-analyser. License agreement is mentioned in the README.)

- scikit-learn (TfidfVectorizer, Ridge, Pipeline, GroupShuffleSplit), pandas, NumPy, re, files, DummyRegressor, mean_absolute_error, mean_squared_error, r2_score,

- AI tools: I used ChatGPT to help debug, understand documentation of scikit-learn, to understand the flow of an AI project and required libraries, and to understand the methodology for the CLEAR dataset.